# Introduction to Pandas

Pandas organizes labeled, tabular data and supports the data-cleaning and exploration work that appears throughout machine learning.

Run the notebook from top to bottom, or begin at any numbered section: each section creates the small objects it uses.

---

## 1. Series: values, labels, and dictionary-like access

A `Series` stores one-dimensional data together with an index of labels.

In [2]:
import numpy as np
import pandas as pd

scores = pd.Series([88, 94, 91, 87], index=["Ada", "Ben", "Chloe", "Diego"], name="quiz_score")
scores

Ada      88
Ben      94
Chloe    91
Diego    87
Name: quiz_score, dtype: int64

In [2]:
scores.values, scores.index

(array([88, 94, 91, 87]), Index(['Ada', 'Ben', 'Chloe', 'Diego'], dtype='str'))

The values and labels are separate pieces of a Series. Labels make it possible to refer to an observation by a meaningful name.

In [3]:
temperatures = pd.Series({"Amarillo": 72, "Canyon": 70, "Hereford": 74}, name="temperature_f")
temperatures["Hereford"]

np.int64(74)

A Series constructed from a dictionary supports dictionary-like lookup while still retaining vectorized numerical operations.

---

## 2. DataFrames: tables and common construction patterns

A `DataFrame` is a labeled table. Columns can have different meanings and data types.

In [4]:
import numpy as np
import pandas as pd

students = pd.DataFrame(
    {
        "name": ["Ada", "Ben", "Chloe"],
        "hours_studied": [5.0, 3.5, 6.0],
        "passed": [True, True, True],
    }
)
students

,name,hours_studied,passed
0,Ada,5.0,True
1,Ben,3.5,True
2,Chloe,6.0,True


In [5]:
measurements = pd.DataFrame(
    [
        {"sample": "A", "mass_g": 1.2, "group": "control"},
        {"sample": "B", "mass_g": 1.5, "group": "treatment"},
        {"sample": "C", "mass_g": 1.1, "group": "control"},
    ]
)
measurements

,sample,mass_g,group
0,A,1.2,control
1,B,1.5,treatment
2,C,1.1,control


A dictionary of equal-length lists is convenient for column-oriented data; a list of dictionaries is convenient for record-oriented data.

In [16]:
rng = np.random.default_rng(42)
simulated = pd.DataFrame(
    rng.normal(loc=0.0, scale=1.0, size=(3, 2)),
    columns=["feature_1", "feature_2"],
)
simulated

,feature_1,feature_2
0,0.304717,-1.039984
1,0.750451,0.940565
2,-1.951035,-1.302180


The fixed random seed makes the simulated example reproducible while preserving the common array-to-table construction pattern.

If you already a few Pandas `Series` objects created, you can also create a DataFrame with those as well:

In [6]:
hours = pd.Series({"Ada": 5.0, "Ben": 3.5, "Chloe": 6.0}, name="hours")
rate = pd.Series({"Ben": 18.0, "Chloe": 20.0, "Diego": 16.0}, name="rate")
employees = pd.DataFrame({"hours": hours, "rate": rate})
employees

,hours,rate
Ada,5.0,NaN
Ben,3.5,18.0
Chloe,6.0,20.0
Diego,NaN,16.0


### Summary information for a DataFrame

`shape`, `dtypes`, `count()`, and `info()` give quick checks of table dimensions, column types, non-missing values, and memory-oriented summary information.

In [7]:
print(simulated.shape)
print(simulated.dtypes)
print(simulated.count())

NameError: name 'simulated' is not defined

In [8]:
employees.info()

<class 'pandas.DataFrame'>
Index: 4 entries, Ada to Diego
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   hours   3 non-null      float64
 1   rate    3 non-null      float64
dtypes: float64(2)
memory usage: 96.0+ bytes


---

## 3. Selecting and filtering data with `.loc`, `.iloc`, and Boolean conditions

Use `.loc` for label-based selection and `.iloc` for position-based selection.

In [9]:
import pandas as pd

city_data = pd.DataFrame(
    {"population": [205871, 16885, 272100, 101849], "elevation": [3605, 3543, 3256, 948]},
    index=["Amarillo", "Canyon", "Lubbock", "Wichita Falls"],
)
city_data

,population,elevation
Amarillo,205871,3605
Canyon,16885,3543
Lubbock,272100,3256
Wichita Falls,101849,948


In [10]:
city_data.loc[["Amarillo", "Lubbock"], ["population", "elevation"]]

,population
Amarillo,205871
Lubbock,272100


In [11]:
city_data.iloc[1:3, :1]

,population
Canyon,16885
Lubbock,272100


`.loc` names the rows and columns to select; `.iloc` counts from zero like ordinary Python positional slicing.

In [13]:
city_data["population"] >= 200000

Amarillo          True
Canyon           False
Lubbock           True
Wichita Falls    False
Name: population, dtype: bool

In [19]:
city_data[city_data["population"] >= 200000]

,population,elevation
Amarillo,205871,3605
Lubbock,272100,3256


A Boolean Series acts as a row filter. Here it keeps cities meeting the stated population condition.

---

## 4. Vectorized operations and label alignment

Pandas applies arithmetic to whole Series or DataFrames at once and matches labels before combining data.

In [20]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
readings = pd.Series(rng.integers(1, 10, size=4), index=["A", "B", "C", "D"], name="reading")
print(readings)
print(readings * 2)
print(np.sqrt(readings))

A    1
B    7
C    6
D    4
Name: reading, dtype: int64
A     2
B    14
C    12
D     8
Name: reading, dtype: int64
A    1.000000
B    2.645751
C    2.449490
D    2.000000
Name: reading, dtype: float64


The multiplication and square root are vectorized: one expression operates on every labeled value without writing a Python loop.

Pandas aligns by label, not by position. Labels present in only one Series produce missing values because there is no matching partner.

In [8]:
hours = pd.Series({"Ada": 5.0, "Ben": 3.5, "Chloe": 6.0}, name="hours")
rate = pd.Series({"Ben": 18.0, "Chloe": 20.0, "Diego": 16.0}, name="rate")
hours * rate

Ada        NaN
Ben       63.0
Chloe    120.0
Diego      NaN
dtype: float64

Methods such as `.add(..., fill_value=0)` make a chosen missing-data rule explicit when aligned labels do not fully overlap.

In [13]:
baseline = pd.Series({"A": 2, "B": 4, "C": 6})
adjustment = pd.Series({"B": 1, "C": 3, "D": 5})
baseline.add(adjustment, fill_value=0)

A    2.0
B    5.0
C    9.0
D    5.0
dtype: float64

### Creating new columns from old columns
You can create new columns from old ones similar to how you would do it with a dictionary:

In [26]:
employees = pd.DataFrame({"hours": hours, "rate": rate})
employees['cost'] = hours * rate
employees

,hours,rate,cost
Ada,5.0,NaN,NaN
Ben,3.5,18.0,63.0
Chloe,6.0,20.0,120.0
Diego,NaN,16.0,NaN


---

## 5. Removing data and inspecting a DataFrame

Use the `drop` method to remove rows or columns from a DataFrame. Explicitly use `index=` and `columns=` arguments when dropping rows or columns so the intent is clear. 

Notice that by default, dropping creates a copy of the DataFrame. It does not modify the original. This is good practice because deleting data is always risky - you may think you don't need that data now, but you might decide later that it was important after all. If you want to override this behavior, you can use `.drop(..., inplace=True)`.

In [22]:
import pandas as pd

experiments = pd.DataFrame(
    {
        "run": ["r1", "r2", "r3"],
        "temperature_c": [20.0, 21.5, 19.5],
        "humidity": [0.45, 0.50, 0.55],
        "operator_note": ["ok", "recheck", "ok"],
    }
).set_index("run")
experiments

,temperature_c,humidity,operator_note
run,,,
r1,20.0,0.45,ok
r2,21.5,0.50,recheck
r3,19.5,0.55,ok


In [28]:
experiments.drop(index="r2", inplace=True)

In [25]:
exper2.drop(columns=["operator_note"])

,temperature_c,humidity
run,,
r1,20.0,0.45
r3,19.5,0.55


In [29]:
experiments

,temperature_c,humidity,operator_note
run,,,
r1,20.0,0.45,ok
r3,19.5,0.55,ok


---

## 6. Reading and writing tabular CSV data

CSV files are a portable way to exchange tabular data. This example creates a temporary file, writes it, and reads it back.

In [21]:
from pathlib import Path
import pandas as pd

sales = pd.DataFrame(
    {
        "product": ["notebook", "pen", "folder"],
        "units": [12, 30, 8],
        "price": [3.50, 1.25, 4.00],
    }
)
sales

,product,units,price
0,notebook,12,3.50
1,pen,30,1.25
2,folder,8,4.00


In [22]:
# Where to put the csv file
csv_path = Path('sales.csv')

# Save the save DataFrame to a csv file
sales.to_csv(csv_path, index=False)  # index=False means the index column is not stored

# Read csv data and store it in the variable restored_sales
restored_sales = pd.read_csv(csv_path)

# Read the regular text for comparison
csv_text = csv_path.read_text(encoding="utf-8")

# Print both so we can compare
print(csv_text)
restored_sales

product,units,price
notebook,12,3.5
pen,30,1.25
folder,8,4.0



,product,units,price
0,notebook,12,3.50
1,pen,30,1.25
2,folder,8,4.00
